# Fast Ray Images (Two Pooch Files)

This notebook fetches the two Zenodo sample files used in this repo and builds one ray-based image per file.

It uses `OctreeRayInterpolator` and picks the faster of:
- `integrate_field_along_rays` (exact segment integral)
- `integrate_field_along_rays_midpoint` (midpoint per segment)


In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pooch

from starwinds_readplt.dataset import Dataset

from batcamp import OctreeInterpolator
from batcamp import OctreeRayInterpolator


### Fetch The Two Pooch Files


In [ ]:
URL = 'https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz'
KNOWN_HASH = 'c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136'
MEMBERS = {
    'SC': 'run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt',
    'IH': 'run-Sun-G2211/IH/IO2/3d__var_4_n00005000.plt',
}

paths = {
    key: Path(
        pooch.retrieve(
            url=URL,
            known_hash=KNOWN_HASH,
            progressbar=False,
            processor=pooch.Untar(members=[member]),
        )[0]
    )
    for key, member in MEMBERS.items()
}

paths


### Ray Image Helpers

Camera: launch rays from the `xmin` side, direction `+x`.


In [ ]:
IMAGE_NY = 3
IMAGE_NZ = 3


def make_x_camera(tree, *, ny=IMAGE_NY, nz=IMAGE_NZ):
    dmin, dmax = tree.domain_bounds(coord='xyz')
    xmin, xmax = float(dmin[0]), float(dmax[0])
    ymin, ymax = float(dmin[1]), float(dmax[1])
    zmin, zmax = float(dmin[2]), float(dmax[2])

    y = np.linspace(ymin, ymax, ny)
    z = np.linspace(zmin, zmax, nz)
    Yg, Zg = np.meshgrid(y, z, indexing='xy')

    x0 = xmin + 1e-6 * (xmax - xmin)
    origins = np.column_stack(
        (
            np.full(Yg.size, x0, dtype=float),
            Yg.ravel(),
            Zg.ravel(),
        )
    )
    direction = np.array([1.0, 0.0, 0.0], dtype=float)
    t_end = (xmax - xmin) * 0.999999

    return origins, direction, t_end, (ymin, ymax, zmin, zmax), Yg.shape


def build_context(path, label):
    t0 = time.perf_counter()
    ds = Dataset.from_file(str(path))
    interp = OctreeInterpolator(ds, ['Rho [g/cm^3]'])
    ray = OctreeRayInterpolator(interp)
    setup_seconds = time.perf_counter() - t0

    print(
        f"{label}: setup={setup_seconds:.3f}s, tree_coord={interp.tree.tree_coord}"
    )

    return {
        'label': label,
        'path': path,
        'interp': interp,
        'ray': ray,
    }


def _integrate_on_camera(ctx, origins, direction, t_end, *, method='exact', chunk_size=4096):
    if method == 'exact':
        return ctx['ray'].integrate_field_along_rays(
            origins,
            direction,
            0.0,
            float(t_end),
            chunk_size=chunk_size,
        )
    if method == 'midpoint':
        return ctx['ray'].integrate_field_along_rays_midpoint(
            origins,
            direction,
            0.0,
            float(t_end),
            chunk_size=chunk_size,
        )
    raise ValueError(f"Unknown method: {method}")


def benchmark_integrators(ctx, *, ny=IMAGE_NY, nz=IMAGE_NZ, chunk_size=4096):
    origins, direction, t_end, _extent_xyz, _image_shape = make_x_camera(ctx['interp'].tree, ny=ny, nz=nz)

    timings = {}
    for method in ('exact', 'midpoint'):
        t0 = time.perf_counter()
        _ = np.asarray(
            _integrate_on_camera(
                ctx,
                origins,
                direction,
                t_end,
                method=method,
                chunk_size=chunk_size,
            ),
            dtype=float,
        )
        timings[method] = time.perf_counter() - t0

    winner = min(timings, key=timings.get)
    print(
        f"{ctx['label']}: rays={origins.shape[0]} ({ny}x{nz}), "
        f"exact={timings['exact']:.3f}s, midpoint={timings['midpoint']:.3f}s, winner={winner}"
    )
    return winner, timings


def render_ray_image(ctx, *, method='exact', ny=IMAGE_NY, nz=IMAGE_NZ, chunk_size=4096):
    origins, direction, t_end, extent_xyz, image_shape = make_x_camera(ctx['interp'].tree, ny=ny, nz=nz)

    t0 = time.perf_counter()
    values = np.asarray(
        _integrate_on_camera(
            ctx,
            origins,
            direction,
            t_end,
            method=method,
            chunk_size=chunk_size,
        ),
        dtype=float,
    ).reshape(image_shape)
    elapsed = time.perf_counter() - t0

    finite_count = int(np.isfinite(values).sum())
    positive_count = int(np.count_nonzero(np.isfinite(values) & (values > 0.0)))

    print(
        f"{ctx['label']}: rays={origins.shape[0]} ({ny}x{nz}), method={method}, full={elapsed:.3f}s, "
        f"finite={finite_count}/{values.size}, positive={positive_count}/{values.size}"
    )

    plot_data = np.where(
        np.isfinite(values) & (values > 0.0),
        np.log10(values),
        np.nan,
    )

    valid = np.isfinite(plot_data)
    if np.any(valid):
        vmin = float(np.nanpercentile(plot_data[valid], 2.0))
        vmax = float(np.nanpercentile(plot_data[valid], 98.0))
    else:
        vmin, vmax = -1.0, 1.0

    ymin, ymax, zmin, zmax = extent_xyz
    fig, ax = plt.subplots(figsize=(7.4, 6.2))
    cmap = plt.colormaps['viridis'].copy()
    cmap.set_bad('lightgray')

    im = ax.imshow(
        plot_data,
        origin='lower',
        extent=[ymin, ymax, zmin, zmax],
        aspect='equal',
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel('Y [R]')
    ax.set_ylabel('Z [R]')
    ax.set_title(f"{ctx['label']}: log10(Integral rho ds), method={method}")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label('log10(Integral rho ds)')
    fig.tight_layout()

    out_png = Path(f"ray_fast_{ctx['label'].lower()}.png")
    fig.savefig(out_png, dpi=180, bbox_inches='tight')
    plt.show()

    return {
        'label': ctx['label'],
        'tree_coord': str(ctx['interp'].tree.tree_coord),
        'method': method,
        'full_seconds': elapsed,
        'finite': finite_count,
        'positive': positive_count,
        'total': int(values.size),
        'png': str(out_png),
        'ny': int(ny),
        'nz': int(nz),
    }


### Build Contexts Once (3x3 Debug Mode)

Build and cache dataset/interpolator/ray objects once per file.


In [ ]:
contexts = {}
for label in ('SC', 'IH'):
    contexts[label] = build_context(paths[label], label)
list(contexts.keys())


### Resolution Control

Change `IMAGE_NY`/`IMAGE_NZ` here and rerun benchmark/render cells.


In [ ]:
IMAGE_NY = 3
IMAGE_NZ = 3
print(f'Resolution set to {IMAGE_NY}x{IMAGE_NZ} ({IMAGE_NY*IMAGE_NZ} rays)')


### Benchmark Integrators (Optional)

Run this cell if you want timing comparisons. It is separate from rendering.


In [ ]:
method_map = {}
timings_map = {}
for label in ('SC', 'IH'):
    winner, timings = benchmark_integrators(contexts[label], ny=IMAGE_NY, nz=IMAGE_NZ, chunk_size=4096)
    method_map[label] = winner
    timings_map[label] = timings
method_map, timings_map


### Render SC And IH Images


In [ ]:
if 'contexts' not in globals():
    raise RuntimeError('Run the context build cell first.')

if 'method_map' not in globals():
    method_map = {'SC': 'exact', 'IH': 'exact'}

results = []
for label in ('SC', 'IH'):
    results.append(
        render_ray_image(
            contexts[label],
            method=method_map.get(label, 'exact'),
            ny=IMAGE_NY,
            nz=IMAGE_NZ,
            chunk_size=4096,
        )
    )

results
